# ByT5 fine-tuning: informal Latin → Cyrillic Mongolian

Runs on a **free Colab T4** (Runtime → Change runtime type → T4 GPU).
Everything (data, checkpoints, results) is stored in Google Drive, so a disconnect
only costs the steps since the last checkpoint — just re-run all cells, training resumes.

Steps: setup → generate synthetic data → train → evaluate (synthetic + real test) → error analysis → try it.

In [ ]:
!nvidia-smi

## 1. Setup
Mount Drive and get the code. Either set `REPO_URL` to your GitHub repo, or upload `mn-translit.zip` (zip of the project folder) to `MyDrive/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/mn-translit-run'   # data, checkpoints, results live here
REPO_URL = ''                                       # e.g. 'https://github.com/<you>/mn-translit.git'
os.makedirs(DRIVE, exist_ok=True)

if not os.path.exists('/content/mn-translit'):
    if REPO_URL:
        !git clone -q $REPO_URL /content/mn-translit
    else:
        !unzip -q /content/drive/MyDrive/mn-translit.zip -d /content/
%cd /content/mn-translit
!ls

In [ ]:
!pip install -q -r requirements.txt

## 2. Synthetic data
Mongolian Wikipedia (~20k articles) + the conversational seed file (oversampled).
`data/real_test.jsonl` sentences are automatically excluded. Data is cached in Drive.

In [ ]:
DATA = f'{DRIVE}/data'
if not os.path.exists(f'{DATA}/train.jsonl'):
    !python generate_pairs.py --hf --max-docs 20000 --extra data/seed_conversational.txt \
        --variants 2 --max-train-pairs 200000 --out-dir $DATA
!wc -l $DATA/*.jsonl
!cp -n data/real_test.jsonl $DATA/ 2>/dev/null; ls $DATA

In [ ]:
import json, itertools
for line in itertools.islice(open(f'{DATA}/train.jsonl', encoding='utf-8'), 8):
    r = json.loads(line); print(r['src'], '\n  ->', r['tgt'])

## 3. Train
fp32 on T4 (fp16 makes ByT5 diverge to NaN). ~5000 steps × 32 examples ≈ 2.5–3 h.
Watch the first log lines: if it/s is much lower than expected, reduce `--max-steps`.
Early stopping on validation CER (greedy decoding during training; beam 4 in evaluation).

In [ ]:
MODEL_DIR = f'{DRIVE}/models/byt5-mn'
!python train/train_byt5.py --model google/byt5-small \
    --train $DATA/train.jsonl --valid $DATA/valid.jsonl --out $MODEL_DIR \
    --batch-size 16 --grad-accum 2 --lr 3e-4 --max-steps 5000 --eval-steps 500 --resume

## 4. Evaluate (2 baselines + model, synthetic test + real test)

In [ ]:
RESULTS = f'{DRIVE}/results'
!python eval/evaluate.py --test $DATA/test.jsonl --real $DATA/real_test.jsonl \
    --model $MODEL_DIR --model-name byt5-small --num-beams 4 --out-dir $RESULTS

In [ ]:
from IPython.display import Markdown
Markdown(open(f'{RESULTS}/results.md', encoding='utf-8').read())

## 5. Error analysis

In [ ]:
!python eval/error_analysis.py $RESULTS/pred_synthetic_test.csv --system byt5-small --top 15

In [ ]:
if os.path.exists(f'{RESULTS}/pred_real_test.csv'):
    !python eval/error_analysis.py $RESULTS/pred_real_test.csv --system byt5-small --top 15

## 6. Try it

In [ ]:
from translit.model import Seq2SeqTransliterator
from baselines import get_baseline
tr = Seq2SeqTransliterator(MODEL_DIR, num_beams=4)
mns = get_baseline('mns5217')
for s in ['za bi margaash ugluu irne', 'bi meeting-ruu orloo', 'unuudur ih zavgvi bailaa',
          'hvvhdvvd surguulidaa yavsan uu', 'margash hed tsagt uulzah ve']:
    print(f'{s}\n  model : {tr(s)}\n  mns   : {mns(s)}')

## 7. (Optional) Upload the model to the Hugging Face Hub
Needed for the HF Spaces demo (`MODEL_ID` env var). Create a write token at hf.co/settings/tokens.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_REPO = '<your-username>/byt5-small-mn-latin2cyrl'
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).push_to_hub(HF_REPO)
AutoTokenizer.from_pretrained(MODEL_DIR).push_to_hub(HF_REPO)